In [1]:
# user defined variables 
# read the data
# strip ontology of unknown terms from DEGs
# iterate a hypergeometric test over categories to get P values
# multiple test correct P values
# store the results in a table

In [2]:
# should we try using also Phenotype_AutoRIF_Predicted_Z-score.txt

In [3]:
import pandas, numpy
import scipy, scipy.stats
import statsmodels, statsmodels.stats, statsmodels.stats.multitest

# user defined variables 

In [4]:
DEGs_file = 'effect_WT_vs_MUTANT_annotated_update.tsv' 
transcriptome_file = 'effect_WT_vs_MUTANT_annotated_update_nocounts.tsv'
ontology_file = 'GO_Biological_Process_GeneRIF_Predicted_Z-score.txt'

# read the data

## read DEGs

In [5]:
degs_def = pandas.read_csv(DEGs_file, sep='\t')
degs_def

,ensembl_gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,external_gene_name,gene_biotype,description,logP,significance
1,ENSDARG00000000001,414.065686,0.017795,0.147620,0.014587,9.038680e-01,9.486372e-01,slc35a5,protein_coding,solute carrier family 35 member A5 [Source:ZFI...,0.022900,Not Significant
2,ENSDARG00000000002,122.646438,-0.456002,0.224308,4.126877,4.220718e-02,1.300113e-01,ccdc80,protein_coding,coiled-coil domain containing 80 [Source:ZFIN;...,0.886019,Not Significant
3,ENSDARG00000000018,1051.035521,0.136785,0.124240,1.212924,2.707538e-01,4.575218e-01,nrf1,protein_coding,nuclear respiratory factor 1 [Source:ZFIN;Acc:...,0.339588,Not Significant
5,ENSDARG00000000019,4589.328732,0.079956,0.051831,2.379725,1.229192e-01,2.709666e-01,ube2h,protein_coding,ubiquitin-conjugating enzyme E2H (UBC8 homolog...,0.567084,Not Significant
6,ENSDARG00000000068,404.006615,0.158394,0.157004,1.017005,3.132306e-01,5.026760e-01,nherf1a,protein_coding,NHERF family PDZ scaffold protein 1a [Source:Z...,0.298712,Not Significant
...,...,...,...,...,...,...,...,...,...,...,...,...
32189,ENSDARG00000117180,38.286079,-1.846321,0.516689,12.373820,4.353958e-04,3.926125e-03,NaN,protein_coding,NaN,2.406036,Downregulated
32190,ENSDARG00000117186,133.301895,-4.196041,0.577328,43.709999,3.808211e-11,1.757338e-09,dab2ipb,protein_coding,DAB2 interacting protein b [Source:ZFIN;Acc:ZD...,8.755145,Downregulated
32191,ENSDARG00000117190,24.432968,-8.067412,1.106920,89.266929,3.449782e-21,4.285986e-19,cadps2,protein_coding,Ca++-dependent secretion activator 2 [Source:Z...,18.367949,Downregulated
32192,ENSDARG00000117199,70.865264,1.712388,0.365906,21.255557,4.019434e-06,7.168895e-05,kctd16a,protein_coding,potassium channel tetramerization domain conta...,4.144548,Upregulated


In [6]:
up_df = degs_def[degs_def['significance'] == 'Upregulated']
down_df = degs_def[degs_def['significance'] == 'Downregulated']

print(up_df.shape)
print(down_df.shape)

print(len(set(up_df['ensembl_gene_id'])))
print(len(set(down_df['ensembl_gene_id'])))

print('DEG set size is 532 + 497 = 1029')

a = up_df['ensembl_gene_id'].to_list()
b = up_df['external_gene_name'].to_list()
c = down_df['ensembl_gene_id'].to_list()
d = down_df['external_gene_name'].to_list()
print(d)

# get rid of repeats and nans
all_degs_ids = a + b + c + d
print(len(all_degs_ids))
all_degs_ids = list(set(all_degs_ids))
print(len(all_degs_ids))
all_degs_ids.remove(numpy.nan)
print(len(all_degs_ids))
numpy.nan in all_degs_ids

(532, 12)
(497, 12)
532
497
DEG set size is 532 + 497 = 1029
['nr4a1', 'ankrd22', 'fam113', 'mybphb', 'col8a1b', 'rnaseh2b', 'desmb', 'ugp2a', 'arhgef7a', 'sult1st6', 'fancl', 'si:ch73-111e15.1', 'znfl2a', 'USP28', 'ZNF385A', 'srpx', 'prg4a', 'slc25a25a', 'dnase1l4.2', 'vipr2', 'hsc70', 'slc4a1a', 'itgb5', 'or6at1', 'uap1l1', 'spo11', 'ern1', 'pgghg', 'golim4b', 'lims2', 'PHIP', 'med7', 'ckmt1', 'abcc6a', 'srrt', 'illr3', 'itga2b', 'gsr', 'tfe3b', 'cbr1l', 'naalad2', 'zgc:109913', nan, 'cops7a', 'ptk2ba', 'si:dkey-23a23.2', 'crema', 'rilpl2', 'ptgr1.1', 'prr15la', 'med29', 'hmox1a', 'hao2', 'tmt1a.2', 'ahsa1a', 'hsp70.1', 'cldn11b', 'fermt3b', 'tnnt2c', 'coq8ab', nan, nan, 'slc25a25b', 'acana', 'gmnn', 'irs2a', 'qdprb.1', 'med28', 'npy1r', 'gdf3', 'fam83fa', 'lgals2b', 'pitrm1', 'fggy', 'zgc:113691', 'zp3b', 'zgc:101562', 'sult3st4', 'noxo1a', 'eef2a.2', 'eef2a.1', 'si:dkey-51e6.1', 'nup205', 'cela1.4', 'setd6', 'sumf1', 'ddx19a', 'elac1', 'fkbp10b', 'PTS', nan, 'si:ch211-283g2.1', 'oc

False

## read ontology

In [7]:
onto_df = pandas.read_csv(ontology_file, sep='\t', header=None, index_col=0)
onto_df

,1,2,3,4,5,6,7,8,9,10,...,193,194,195,196,197,198,199,200,201,202
0,,,,,,,,,,,,,,,,,,,,,
AMP_biosynthetic_process_(GO:0006167),NaN,ENSDARG00000103096,ENSDARG00000110863,ENSDARG00000103890,zgc:174574,ENSDARG00000078489,ENSDARG00000112484,ENSDARG00000113576,ENSDARG00000098090,sertad2b,...,ENSDARG00000110111,hsp70l,gpr35.1,si:zfos-364h11.1,arhgef33,march5,si:ch211-168d1.3,npffr1l1,cyr61l2,NaN
ATP_biosynthetic_process_(GO:0006754),NaN,slc15a1b,slc15a1a,ENSDARG00000110624,ace2,ENSDARG00000112368,cel.1,asah2,cldn15la,ogal,...,ENSDARG00000112419,ENSDARG00000110230,alpi.2,ENSDARG00000079560,lgals2b,ENSDARG00000101645,cdhr5b,epdl2,slc5a9,NaN
ATP_metabolic_process_(GO:0046034),NaN,slc15a1b,slc15a1a,ENSDARG00000110624,ace2,ENSDARG00000112368,cel.1,asah2,cldn15la,ogal,...,ENSDARG00000112419,ENSDARG00000110230,alpi.2,ENSDARG00000079560,lgals2b,ENSDARG00000101645,cdhr5b,epdl2,slc5a9,NaN
BMP_signaling_pathway_(GO:0030509),NaN,dlgap5,cnbpa,hnrnph1l,rtca,skp2,khdrbs1a,ENSDARG00000103417,cdca5,cstf3,...,ENSDARG00000113623,ENSDARG00000112653,e2f8,hnrnpa0b,rac1a,lrwd1,hnrnpa0l,ppm1g,si:ch211-237c6.4,NaN
B_cell_activation_(GO:0042113),NaN,ENSDARG00000103096,ENSDARG00000110742,ENSDARG00000110863,ENSDARG00000096477,ENSDARG00000103890,ENSDARG00000099504,ENSDARG00000116169,ENSDARG00000116285,ENSDARG00000105716,...,ENSDARG00000109481,ENSDARG00000112772,ENSDARG00000113982,ENSDARG00000115123,ENSDARG00000113502,ENSDARG00000113986,ENSDARG00000113060,ENSDARG00000114152,ENSDARG00000115241,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vesicle_cytoskeletal_trafficking_(GO:0099518),NaN,krcp,caprin2,si:ch211-137a8.2,ccnj,mastl,pask,mis18bp1,si:dkey-25o16.4,smc4,...,ENSDARG00000112534,p4hb,tnpo2,ccdc14,osbpl7,macc1,rrm2,dtl,lrmp,NaN
vesicle_docking_(GO:0048278),NaN,napba,eno2,nptna,fam43b,sv2a,tapbpl,ccdc175,nat8l,si:dkey-205h13.1,...,grin2bb,crhb,grid2ipb,nudt4b,slc6a11b,tmem59l,ENSDARG00000105209,spata4,cx43,NaN
vesicle_fusion_(GO:0006906),NaN,lats2,ENSDARG00000112921,ccdc175,chd4a,slc44a1b,itgav,sox5,elmsan1b,ENSDARG00000110183,...,dnah9,dnah2,prdm5,dnah5,lamc1,spag7,ENSDARG00000087299,kif18a,tpra,NaN


## read transcriptome genes

In [8]:
transcriptome_df = pandas.read_csv(transcriptome_file, sep='\t')
transcriptome_df

,ensembl_gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,external_gene_name,gene_biotype,description,logP,significance
1,ENSDARG00000000001,413.885087,0.032493,0.138647,0.055019,0.814550,0.932030,slc35a5,protein_coding,solute carrier family 35 member A5 [Source:ZFI...,0.030570,Not Significant
2,ENSDARG00000000002,122.521472,-0.441197,0.211024,4.366879,0.036644,0.169364,ccdc80,protein_coding,coiled-coil domain containing 80 [Source:ZFIN;...,0.771178,Not Significant
3,ENSDARG00000000018,1050.869463,0.150754,0.118731,1.613173,0.204046,0.484570,nrf1,protein_coding,nuclear respiratory factor 1 [Source:ZFIN;Acc:...,0.314644,Not Significant
4,ENSDARG00000000018,1050.869463,0.150754,0.118731,1.613173,0.204046,0.484570,nrf1,protein_coding,nuclear respiratory factor 1 [Source:ZFIN;Acc:...,0.314644,Not Significant
5,ENSDARG00000000019,4590.385005,0.093903,0.055966,2.814977,0.093388,0.307582,ube2h,protein_coding,ubiquitin-conjugating enzyme E2H (UBC8 homolog...,0.512039,Not Significant
...,...,...,...,...,...,...,...,...,...,...,...,...
57771,ENSDARG00000117202,0.000000,NaN,NaN,NaN,NaN,NaN,pimr144,protein_coding,"Pim proto-oncogene, serine/threonine kinase, r...",NaN,Not Significant
57772,ENSDARG00000117202,0.000000,NaN,NaN,NaN,NaN,NaN,pimr144,protein_coding,"Pim proto-oncogene, serine/threonine kinase, r...",NaN,Not Significant
57773,ENSDARG00000117204,2.635626,-1.864834,1.289099,2.099393,0.147358,0.404960,NaN,protein_coding,NaN,0.392588,Not Significant
57774,ENSDARG00000117205,6.043247,-0.021726,0.959914,0.002536,0.959839,0.980656,atp5meb,protein_coding,ATP synthase membrane subunit eb [Source:ZFIN;...,0.008483,Not Significant


# strip ontology of unknown terms from DEGs

In [9]:
a = transcriptome_df['ensembl_gene_id'].to_list()
b = transcriptome_df['external_gene_name'].to_list()
transcriptome_names = a + b
print(len(transcriptome_names))
print(transcriptome_names[:20])
transcriptome_names = list(set(transcriptome_names))
print(len(transcriptome_names))

115550
['ENSDARG00000000001', 'ENSDARG00000000002', 'ENSDARG00000000018', 'ENSDARG00000000018', 'ENSDARG00000000019', 'ENSDARG00000000068', 'ENSDARG00000000068', 'ENSDARG00000000069', 'ENSDARG00000000069', 'ENSDARG00000000069', 'ENSDARG00000000069', 'ENSDARG00000000086', 'ENSDARG00000000086', 'ENSDARG00000000086', 'ENSDARG00000000086', 'ENSDARG00000000103', 'ENSDARG00000000103', 'ENSDARG00000000103', 'ENSDARG00000000103', 'ENSDARG00000000142']
50635


In [10]:
onto_container = {}
for category, gene_list in onto_df.head(5).iterrows():
    #print(category)
    local_list_of_genes = gene_list.to_list()
    new_list = []
    for element in local_list_of_genes:
        if element in transcriptome_names:
            new_list.append(element)
    onto_container[category] = new_list
    
    print(len(local_list_of_genes), len(new_list))
    print(local_list_of_genes[:5])
    print(new_list[:5])
    print()            

202 163
[nan, 'ENSDARG00000103096', 'ENSDARG00000110863', 'ENSDARG00000103890', 'zgc:174574']
['ENSDARG00000103096', 'ENSDARG00000110863', 'ENSDARG00000103890', 'ENSDARG00000078489', 'ENSDARG00000112484']

202 169
[nan, 'slc15a1b', 'slc15a1a', 'ENSDARG00000110624', 'ace2']
['slc15a1b', 'ENSDARG00000110624', 'ace2', 'ENSDARG00000112368', 'cel.1']

202 169
[nan, 'slc15a1b', 'slc15a1a', 'ENSDARG00000110624', 'ace2']
['slc15a1b', 'ENSDARG00000110624', 'ace2', 'ENSDARG00000112368', 'cel.1']

202 191
[nan, 'dlgap5', 'cnbpa', 'hnrnph1l', 'rtca']
['dlgap5', 'cnbpa', 'hnrnph1l', 'rtca', 'skp2']

202 184
[nan, 'ENSDARG00000103096', 'ENSDARG00000110742', 'ENSDARG00000110863', 'ENSDARG00000096477']
['ENSDARG00000103096', 'ENSDARG00000110742', 'ENSDARG00000110863', 'ENSDARG00000096477', 'ENSDARG00000103890']



In [11]:
%%time 
# this is a long step, it takes circa 50 secs in an M1 chip

onto_container = {}
for category, gene_list in onto_df.iterrows():
    #print(category)
    local_list_of_genes = gene_list.to_list()
    new_list = []
    for element in local_list_of_genes:
        if element in transcriptome_names:
            new_list.append(element)
    onto_container[category] = new_list
    
    #print(len(local_list_of_genes), len(new_list))
    #print(local_list_of_genes[:5])
    #print(new_list[:5])
    #print()            

CPU times: user 48.7 s, sys: 101 ms, total: 48.8 s
Wall time: 48.8 s


In [12]:
all_genes_in_annotation = []
for element in onto_container.keys():
    for gene in onto_container[element]:
        all_genes_in_annotation.append(gene)
all_genes_in_annotation = list(set(all_genes_in_annotation))
print(len(all_genes_in_annotation))

12119


In [13]:
# check how many DEGs are in the annotation


In [14]:
print(len(all_degs_ids))
count = 0
for element in all_degs_ids:
    if element in all_genes_in_annotation:
        count = count + 1
print(count)

1734
467


# iterate a hypergeometric test over categories to get P values

In [15]:
a = 13588
b = 611
c = 59
d = 19
scipy.stats.hypergeom(a, b, c).sf(d-1)

4.989682834519699e-12

In [16]:
a = 12119 # total number of genes in the annotation, 12119 
c = 467   # DEGs with at least on category in the ontology

all_pvalues = []
sampling_categories = list(onto_container.keys())
sampling_categories.sort()
for category in sampling_categories:
    b = len(onto_container[category]) # b, size of the category
    d = 0 # d, number of genes in category that are DEGs
    hits = []
    for element in onto_container[category]:
        if element in all_degs_ids:
            d = d + 1
            hits.append(element)
    # test
    p = scipy.stats.hypergeom(a, b, c).sf(d-1)
    all_pvalues.append(p)
    if p < 0.05:
        print(category, b, d, p)
        print(', '.join(hits))
        print()

L-serine_metabolic_process_(GO:0006563) 173 12 0.0354975975457036
lats2, sec31a, krt8, ENSDARG00000116888, scn1ba, gli1, yap1, dusp6, metrn, ENSDARG00000115103, abl1, ENSDARG00000109279

Schwann_cell_development_(GO:0014044) 176 16 0.0012672764830885547
itgb5, ENSDARG00000109292, elovl7a, hcfc1a, ctcf, stxbp6l, ENSDARG00000112083, chp2, tuft1a, pla1a, zgc:174888, ENSDARG00000113261, ENSDARG00000109274, ENSDARG00000115952, ENSDARG00000114975, erbb3b

calcium_ion_homeostasis_(GO:0055074) 166 12 0.026960228910854485
lats2, krt8, lgals9l1, rhcgb, dusp6, caspb, acana, si:ch73-288o11.5, ENSDARG00000114130, ENSDARG00000074749, erbb3b, si:dkey-192d15.2

carboxylic_acid_biosynthetic_process_(GO:0046394) 182 12 0.04908670128654841
mapk14a, eif4bb, stm, tspan15, kmt5ab, add3b, cd2bp2, ENSDARG00000103441, ENSDARG00000113285, srrt, nono, znfl2a

carboxylic_acid_catabolic_process_(GO:0046395) 182 12 0.04908670128654841
mapk14a, eif4bb, stm, tspan15, kmt5ab, add3b, cd2bp2, ENSDARG00000103441, ENSDARG

# multiple test correct P values

In [17]:
#methods = ['bonferroni', 'sidak', 'holm-sidak', 'holm', 'simes-hochberg', 'hommel', 'fdr_bh', 'fdr_by', 'fdr_tsbh', 'fdr_tsbky']
methods = ['bonferroni', 'fdr_bh']

for the_method in methods:
    results = statsmodels.stats.multitest.multipletests(all_pvalues, method=the_method)
    print(the_method)
    for i in range(len(results[0])):
        if results[1][i] < 0.05:
            print(i, results[0][i], results[1][i], sampling_categories[i])
    print()

bonferroni
204 True 0.014322970330422909 central_nervous_system_myelination_(GO:0022010)

fdr_bh
204 True 0.014322970330422909 central_nervous_system_myelination_(GO:0022010)
557 True 0.03921023003221469 myelination_in_peripheral_nervous_system_(GO:0022011)

